In [5]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
import pickle

with open('model_with_imputer.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

model = loaded_data['model']
imputer = loaded_data['imputer']

df_demographic = pd.read_csv("sample_data/demographic.csv")
df_hospital = pd.read_csv("sample_data/hospital.csv")
df_physiological = pd.read_csv("sample_data/physiological.txt", delimiter='\t')
df_severity = pd.read_json("sample_data/severity.json")

income_mapping = {
    '$11-$25k': 1,
    'under $11k': 0,
    '$25k-$50k': 2,
    '$50k-$75k': 3,
    'above $75k': 4
}
df_demographic['inntekt'] = df_demographic['inntekt'].map(income_mapping)
df_severity = df_severity.iloc[:, 0:-1].explode(list(df_severity.columns[2:-1]))
df_severity.reset_index(drop=True, inplace=True)
df_severity=df_severity.sort_values(by=['pasient_id'], ignore_index=True)

df = pd.merge(df_severity, df_demographic, on='pasient_id', how='inner')
df = pd.merge(df, df_hospital, on='pasient_id', how='inner')
df = pd.merge(df, df_physiological, on='pasient_id', how='inner')

pasient_id_df = df[['pasient_id']].copy()




df = df.drop_duplicates()
df = df[df['alder'] >= 0]
df = df.drop([
   'sykdomskategori_id', 'sykdomskategori'
], axis=1)

df['serumalbumin'].fillna(3.5, inplace=True)
df['lungefunksjon'].fillna(333.3, inplace=True)
df['bilirubin'].fillna(1.01, inplace=True)
df['kreatinin'].fillna(1.01, inplace=True)
df['blodurea_nitrogen'].fillna(6.51, inplace=True)
df['hvite_blodlegemer'].fillna(9, inplace=True)
df['urinmengde'].fillna(2502, inplace=True)


df.drop(['adl_pasient', 'adl_stedfortreder'], axis=1, inplace=True)
df.drop(['dnr_status'], axis=1, inplace=True)
df.drop('bilirubin', axis=1, inplace= True)
df.drop('dødsfall', axis = 1, inplace = True)
df.drop('sykehusdød', axis=1, inplace = True)
df.drop('pasient_id', axis=1, inplace = True)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
categorical_columns = ['sykdom_underkategori', 'kreft', 'kjønn', 'etnisitet']
ohe.fit(df[categorical_columns])
df = pd.concat([df.drop(columns=categorical_columns).reset_index(drop=True), 
                     ohe.transform(df[categorical_columns]).reset_index(drop=True)], axis=1)


def clip_column(dataframe, column_name, lower_quantile=None, upper_quantile=None):
    lower_bound = dataframe[column_name].quantile(lower_quantile) if lower_quantile is not None else None
    upper_bound = dataframe[column_name].quantile(upper_quantile) if upper_quantile is not None else None
    
    dataframe[column_name] = dataframe[column_name].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

df = clip_column(df, "urinmengde", lower_quantile=0.25, upper_quantile=0.75)
df = clip_column(df, "kreatinin", upper_quantile=0.95)
df = clip_column(df, "hvite_blodlegemer", upper_quantile=0.95)
df = clip_column(df, "blodurea_nitrogen", upper_quantile=0.75)
df = clip_column(df, "glukose", upper_quantile=0.75)

df['tidlig_overlevelsesestimat_gjennomsnitt'] = (df['lege_overlevelsesestimat_2mnd'] + df['overlevelsesestimat_6mnd']) / 2
df['sen_overlevelsesestimat_gjennomsnitt'] = (df['lege_overlevelsesestimat_6mnd'] + df['overlevelsesestimat_2mnd']) / 2
df['helsetilstand'] = df['fysiologisk_score'] * df['koma_score']

df_imputed = imputer.fit_transform(df)
df = pd.DataFrame(df_imputed, columns=df.columns)

predictions = model.predict(df)

predictions_df = pd.DataFrame(predictions, columns=['Predicted_Oppholdslengde'])
results_df = pd.concat([pasient_id_df.reset_index(drop=True), predictions_df.reset_index(drop=True)], axis=1)
results_df.to_csv("predictions.csv", index=False)

/var/folders/1h/__q6rdbs1lqc2l3nlfz9t72m0000gn/T/ipykernel_94972/2864010826.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['serumalbumin'].fillna(3.5, inplace=True)
/var/folders/1h/__q6rdbs1lqc2l3nlfz9t72m0000gn/T/ipykernel_94972/2864010826.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always